# Rerank RAG — Os Sertões

Este notebook usa a mesma base Python do projeto. O modo inicial é **simulado**: documentos sintéticos, vetores programados e cliente de LLM falso. Nenhuma chamada à OpenAI é feita.

Os textos da simulação são fictícios e não são respostas sobre o livro. A execução real fica para depois da aprovação.

## 1. Ambiente e modo de execução

Selecione o kernel Python (Rerank RAG). `MODO = "simulado"` não precisa de chave nem de baixar o PDF ou o modelo. `"somente_busca"` usa o índice real local sem geração; `"real"` usa a OpenAI e só deve ser escolhido depois da aprovação e da configuração local.

In [ ]:
from pathlib import Path
import sys

raiz = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "rerank_rag.py").is_file() and (p / "simulacao.py").is_file()),
    None,
)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta Rerank-RAG.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from rag_base import (
    baixar_pdf, extrair_paginas, dividir_trechos,
    carregar_perguntas, montar_contexto, ler_json, sha256,
)
from rerank_rag import RerankConfig as Config, RerankRAG
from simulacao import AVISO, executar_simulacao

# Manter simulado até a aprovação para os testes reais.
MODO = "simulado"
if MODO not in ("simulado", "somente_busca", "real"):
    raise ValueError("Escolha simulado, somente_busca ou real.")
config = Config.carregar(raiz)
rag = None if MODO == "simulado" else RerankRAG(raiz, config)
print("Modo:", MODO)
if MODO == "simulado":
    print(AVISO)
else:
    print("Embedding local:", config.embedding_model)

## 2. PDF e texto da obra

No modo simulado esta etapa é pulada; o teste usa documentos sintéticos identificados como `fixture://`.

Para uma execução real, o corpus usa as páginas físicas 33 a 653 da edição configurada. A página impressa equivale à página PDF menos 30. Os relatórios simulados usam páginas fictícias e não devem ser consultados como referências da obra.

In [ ]:
if MODO == "simulado":
    print("Simulação: não lê nem baixa o PDF. As fontes são documentos fictícios.")
else:
    pdf = baixar_pdf(config, raiz)
    paginas = extrair_paginas(pdf, config)
    print("Hash do PDF:", sha256(pdf))
    print("Páginas com texto:", len(paginas))
    for posicao in (0, len(paginas) // 2, len(paginas) - 1):
        pagina = paginas[posicao]
        print(f"PDF p. {pagina['pdf_page']} / livro p. {pagina['printed_page']}")
        print(pagina["text"][:500])

## 3. Divisão em trechos

O teste simulado usa a função real de divisão com um tokenizer simples de teste. Os embeddings são vetores definidos por um mapa explícito entre perguntas e documentos fictícios; isso testa a integração, sem avaliar qualidade semântica.

Na execução real, o E5 usa janelas de até 350 tokens e 50 de sobreposição, com os prefixos `passage: ` e `query: `.

In [ ]:
if MODO == "simulado":
    print("A simulação usa o divisor de trechos com tokenizer e documentos de teste.")
else:
    amostra = dividir_trechos(paginas[:2], rag.modelo.tokenizer, config)
    for trecho in amostra[:3]:
        print(trecho["id"], trecho["text"][:250], "\n")

## 4. Índice e recarga

A simulação cria um índice sintético em pasta temporária, recarrega-o no formato de produção e o remove ao terminar. O índice do livro permanece intacto. Conexões de rede e a construção de um cliente OpenAI real são bloqueadas durante o cenário.

In [ ]:
if MODO == "simulado":
    resultado_simulado = executar_simulacao(raiz, nome="notebook-simulacao")
    execucao_simulada = ler_json(resultado_simulado)
    print(execucao_simulada["aviso"])
    print("Trechos sintéticos:", execucao_simulada["indice"]["total_trechos"])
    print("Chamadas simuladas:", execucao_simulada["chamadas_simuladas"])
    print("API real executada:", execucao_simulada["api_real_executada"])
else:
    manifesto = rag.indexar()
    print("Trechos:", manifesto["total_trechos"])
    print("Dimensões:", manifesto["dimensoes"])
    recarregado = RerankRAG(raiz, config)
    recarregado.carregar_indice()
    assert len(recarregado.trechos) == manifesto["total_trechos"]
    print("Índice real recarregado.")

## 5. Reranking

A busca seleciona candidatos por cosseno. O CrossEncoder reavalia pares pergunta/trecho; somente os melhores entram no contexto. O relatório registra a ordem antes e depois, pontuações e tempo.

In [ ]:
perguntas = carregar_perguntas(raiz)
pergunta = perguntas[0]["pergunta"]
if MODO == "simulado":
    resultados = execucao_simulada["resultados"][0]["recuperados"]
else:
    resultados = rag.buscar(pergunta)
contexto, fontes = montar_contexto(resultados, config.max_context_chars)
print(pergunta)
print("Trechos no contexto:", len(fontes))
print(contexto[:1500])
if MODO == "simulado":
    print(execucao_simulada["resultados"][0]["recuperacao"])
else:
    print(rag.ultima_busca)


## 6. Resposta simulada

O cliente falso retorna um texto marcado como SIMULAÇÃO e uma citação da página fictícia presente no contexto. São conferidos o contrato da chamada, o tratamento da resposta e as referências. Isso não equivale a testar um modelo de linguagem real.

In [ ]:
if MODO == "simulado":
    resposta = execucao_simulada["resultados"][0]
    print(resposta["resposta"])
    print("Citações fictícias:", resposta["citacoes"])
elif MODO == "real":
    resposta = rag.responder(pergunta)
    print(resposta["resposta"])
    print("Conferência de citações:", resposta["citacoes"])
else:
    print("Somente busca: geração não executada.")

## 7. Cinco perguntas e exportação

No modo simulado, Q1 a Q5 geram JSON e Markdown com `simulado: true`, `api_real_executada: false`, tokens e custo como `null`. O cenário sem evidência verifica que nenhuma geração é tentada quando o filtro não retorna trechos.

In [ ]:
if MODO == "simulado":
    resultado_path = resultado_simulado
else:
    resultado_path = rag.avaliar(
        somente_busca=MODO == "somente_busca",
        nome="notebook-respostas" if MODO == "real" else "notebook-busca",
    )
execucao = ler_json(resultado_path)
print("Arquivo:", resultado_path)
for item in execucao["resultados"]:
    print("\n", item["id"], item["modo"])
    print(item["resposta"] or "Somente recuperação.")
if MODO == "simulado":
    negativo = execucao["cenario_sem_evidencia"]
    print("\nCenário sem evidência:", negativo["resposta"])
    assert not negativo["geracao_simulada"]
    assert not execucao["api_real_executada"]

## 8. Revisão e próximos passos

Execute `python rerank_rag.py simular` para reproduzir a simulação para repetir a validação offline.

Os testes cobrem também erros simulados de credencial, limite da API, conexão, resposta incompleta/vazia e citações inválidas. As respostas fictícias não devem receber notas de qualidade literária.

Depois da aprovação, será possível configurar o `.env`, escolher `MODO = "real"` e revisar as cinco respostas reais com a rubrica. Essa etapa continua pendente no `resume.md` e no checklist.